In [2]:
import os

from openai import OpenAI

OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

client = OpenAI(api_key=OPENAI_API_KEY)

completion = client.chat.completions.create(
    model='gpt-3.5-turbo-0125',
    messages=[{'role': 'user', 'content': 'hi'}],
    temperature=0.0,
)

In [3]:
import json

with open('./res/reviews.json', 'r') as f:
    review_list = json.load(f)

In [4]:
import datetime
import re

def preprocess_reviews(path='./res/reviews.json'):
    with open(path, 'r', encoding='utf-8') as f:
        review_list = json.load(f)

    reviews_good, reviews_bad = [], []

    current_date = datetime.datetime.now()
    date_boundary = current_date - datetime.timedelta(days=6*30)

    for r in review_list:
        review_date_str = r.get('date', '')
        # 「YYYY年MM月」形式の日付をパース
        m = re.match(r'(\d{4})年(\d{1,2})月', review_date_str)
        if m:
            review_date = datetime.datetime(int(m.group(1)), int(m.group(2)), 1)
        else:
            review_date = current_date

        if review_date < date_boundary:
            continue

        if r['star'] == 5:
            reviews_good.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')
        else:
            reviews_bad.append('[REVIEW_START]' + r['review'] + '[REVIEW_END]')

    reviews_good_text = '\n'.join(reviews_good)
    reviews_bad_text = '\n'.join(reviews_bad)

    return reviews_good_text, reviews_bad_text

good, bad = preprocess_reviews()

In [5]:
PROMPT_BASELINE = f"""以下の宿泊施設のレビューを5行くらいのボリュームで、まとめてください："""

In [ ]:
reviews, _ = preprocess_reviews(path='./res/reviews.json')

def summarize(reviews, prompt, temperature=0.0, model='gpt-3.5-turbo-0125'):
    prompt = prompt + '\n\n' + reviews

    completion = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=temperature
    )

    return completion

In [8]:
gpt_3_5_result = summarize(reviews, PROMPT_BASELINE).choices[0].message.content
print(gpt_3_5_result)

1. 2家族10人で利用しました。和室湖畔側のお部屋は景色も良く清潔感あふれるお部屋で、夕食朝食ともに大満足でした。子供達も屋上スパに大満足で、忙しい日常から離れて楽しい旅行になりました。

2. 16時前にチェックインしたのですが、18時台のお食事は満席でした。湖が見える良い部屋だったため20時の花火はお部屋で見たかったので、19時のお食事だと急かされる感じがして残念でした。夕食も朝食も種類が豊富で、天空スパは早朝に入り、夜が明けていく感じがステキでした。

3. 急な骨折で静養が必要だったが、スタッフの対応が素晴らしく、和室を用意してくれたり、食事やサービスも助かりました。眺めの良いお部屋で過ごし、スタッフの笑顔に感謝しています。

4. 露天風呂が素晴らしく、朝早くに入るとしらじら夜が明けて湖や山が幻想的でした。冬の阿寒湖を楽しんだ形で、温泉も良く、料理の品数も多くて満足できました。

5. 夕食、朝食のバイキングの品揃えが素晴らしく、どれを食べても美味しかったです。屋上の天空露天風呂も最高で、また利用したいと思える宿でした。


In [ ]:
gpt_4o_result = summarize(reviews, PROMPT_BASELINE, model='gpt-4-turbo-2024-04-09').choices[0].message.content
print(gpt_4o_result)

レビューのまとめ：

1. 宿泊客は全体的に食事の品数と質に大満足しており、特にバイキングスタイルの食事が好評である。
2. 天空スパや屋上の露天風呂が特に評価が高く、景色を楽しみながらリラックスできる点が魅力的とされている。
3. スタッフの対応が親切で、特に急な要望にも柔軟に対応してくれる点が好印象を与えている。
4. 部屋の清潔さや景観も評価が高く、特に湖側の部屋からの眺めが好評。
5. 一部のレビューでは、チェックイン時間や食事時間の混雑に関する不満が見られるが、全体的には満足度が高い宿泊体験が報告されている。


In [10]:
def g_eval_summary(document, summary):
    """
    G-Eval 논문의 프롬프트를 사용하여 요약문의 일관성(Coherence)을 평가합니다.
    """
    
    system_prompt = "You are an expert evaluator for text summarization."
    
    evaluation_prompt = f"""
    You will be given one summary written by a person and one introduction to a news article.
    Your task is to rate the summary on the 'Coherence' metric.

    Evaluation Criteria:
    Coherence (1-5): The collective quality of all sentences. A high-quality summary should be well-structured and organized, and not just a heap of related sentences.

    Evaluation Steps:
    1. Read the news article carefully and identify the main topic and key points.
    2. Read the summary and compare it to the news article.
    3. Check if the summary presents points in a logical and coherent order.
    4. Assign a score for coherence on a scale of 1 to 5.

    Source Text: {document}
    Summary: {summary}

    Output Format:
    Reasoning: (A brief explanation for the score)
    Score: (Only the integer score between 1 and 5)
    """

    # 2. OpenAI API 호출 (GPT-4 권장)
    response = client.chat.completions.create(
        model="gpt-5.2", # 또는 "gpt-4"
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": evaluation_prompt}
        ],
        temperature=0, # 평가의 일관성을 위해 0으로 설정
        seed=42
    )

    return response.choices[0].message.content

In [11]:
eval_gpt_3_5 = g_eval_summary(document=reviews, summary=gpt_3_5_result)
print("GPT-3.5 Evaluation:\n", eval_gpt_3_5)

GPT-3.5 Evaluation:
 Reasoning: The summary is organized as a numbered list that tracks several different guest reviews. Within each numbered item, the sentences generally flow logically (room/food/spa impressions, then overall evaluation). However, as a whole it reads more like separate mini-summaries stitched together rather than a single unified, well-structured summary, and item 4 slightly mixes two different review threads (sunrise open-air bath and winter Akanko/food) which weakens overall coherence. Still, it remains understandable and mostly well ordered.  
Score: 4


In [12]:
eval_gpt_4o = g_eval_summary(document=reviews, summary=gpt_4o_result)
print("GPT-4o Evaluation:\n", eval_gpt_4o)

GPT-4o Evaluation:
 Reasoning: 箇条書きで「食事→スパ→スタッフ→部屋→混雑の不満」という流れが整理されており、全体として読みやすく構造化されています。各文も同じ対象（宿泊レビューの総括）に一貫しており、話題の飛躍や文同士のつながりの悪さはほぼありません。ニュース記事型ではなくレビュー集合の要約として自然な構成です。  
Score: 5


In [13]:
PROMPT_ONE_SHOT = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。
    
以下の宿泊施設のレビューを要約してください："""

In [14]:
gpt_3_5_1shot_result = summarize(reviews, PROMPT_ONE_SHOT).choices[0].message.content
print(gpt_3_5_1shot_result)

1. 2家族10人で利用しました。和室湖畔側のお部屋は景色も良く清潔感あふれるお部屋でした。夕食朝食ともに大満足でした。子供達が屋上スパに大大大満足でした。いつも忙しく子供との時間もとれて無かったので。とても良い旅行になりました。ありがとう感謝です。
2. 16時前にチェックインしたのですが既に18時台のお食事は満席でした。湖が見える良い部屋だったため20時の花火はお部屋で見たかったので、19時のお食事だとせっかくのバイキングも何となく急かされて、途中でお部屋に戻ってからまた食事席に着くのもなぁと…少し残念でした。夕食も朝食も種類が沢山あって、選ぶのが楽しかったです。天空スパは早朝に入りましたが、段々夜が明けていく感じがステキでしたです。
3. そちらに伺う3日ほど前に、骨折をしてしまい、安静にと言う事で、急なお話だったにも関わらず、早めにお部屋に入れてくださり、雪も降っていたので、お陰様で、静養する事が出来ました骨折の原因がベッドだったので、ベッド恐怖症になり、和室が空いていたらお願いしたいと、勝手な申し出にも対応してくださり、食事も、カートをお借りしたりして、助かりました　本当に心から感謝しておりますm(_ _)m本来なら、オンネトーと、阿寒湖氷上歩きのツアーを申し込んでいましたが、動けないので、行く事も叶いませんでしたその代わり、眺めの良いお部屋で、皆さんが湖上で遊んでいるのを見たり、夜には、花火が見えて、楽しかったです急な事にも、関わらず、笑顔で対応してくたざったスタッフのお姉さん、まだお若いと思うのですが、素晴らしい対応していただき、本当にありがとうございました本当に助かりました治りましたら、また、いずれそちらにお世話になりたいと強く思いましたです。


In [15]:
eval_gpt_3_5_1shot = g_eval_summary(document=reviews, summary=gpt_3_5_1shot_result)
print("GPT-3.5 Evaluation:\n", eval_gpt_3_5_1shot)

GPT-3.5 Evaluation:
 Reasoning: The “summary” is essentially three full review excerpts copied verbatim and listed as 1–3, rather than a synthesized, unified summary. Each numbered section is internally understandable, but the overall text lacks integration and consistent structure, includes redundant/awkward sentence endings (“ステキでしたです”, “思いましたです”), and contains run-on sentences and informal fragments that hurt readability. As a collection it’s only loosely organized (by reviewer) and not coherently summarized into a single narrative.
Score: 2


In [24]:
example_reviews = """[REVIEW_START]部屋は広くて快適でした。防音もしっかりしていて、隣の音が全く気になりませんでした。[REVIEW_END]
[REVIEW_START]駅から近くてアクセスが便利でした。フロントの方も親切で、チェックインがスムーズでした。[REVIEW_END]
[REVIEW_START]朝食のパンが美味しかったです。種類も豊富で毎朝楽しみでした。また泊まりたいです。[REVIEW_END]
[REVIEW_START]部屋からの眺めが最高でした。清掃も行き届いていて、気持ちよく過ごせました。[REVIEW_END]"""

example_summary = """部屋が広く清潔で、防音もしっかりしているという評価が多いです。駅からのアクセスが便利で、フロントの対応も親切だという声が多く見られます。朝食はパンを中心に種類が豊富で美味しいと好評です。再訪を希望する声も多く、全体的に満足度の高い宿泊施設という評価です。"""

prompt_1shot = f"""あなたは要約の専門家です。宿泊施設のユーザーレビューが与えられた際に、それを要約することがあなたの目標です。

要約結果は以下の条件を満たす必要があります：
1. すべての文は必ず丁寧語（です・ます調）で終わること。
2. 宿泊施設を紹介するトーン＆マナーで作成してください。
  2-1. 良い例
    a) 全体的に良い宿泊施設で、防音も問題なかったという評価です。
    b) 再訪予定だという声が多く見られます。
  2-2. 悪い例
    a) 良い宿泊施設で、防音も問題ありませんでした。
    b) 再訪予定です。
3. 要約結果は最低2文、最大5文で作成してください。
4. 個別のレビューをそのまま書くのではなく、複数のレビューの傾向をまとめて「〜という評価です」「〜という声が多いです」のように書いてください。
5. 要約は「施設・部屋 → 食事 → サービス・スタッフ → 総合評価」の順で構成してください。

以下はレビューと要約の例です。
例のレビュー：
{example_reviews}
例の要約結果：
{example_summary}
    
以下の宿泊施設のレビューを要約してください："""

prompt_1shot_result = summarize(reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content

eval_gpt_3_5_1shot = g_eval_summary(document=reviews, summary=prompt_1shot_result)

print("Summary:\n", prompt_1shot_result)
print("\nGPT-3.5 Evaluation:\n", eval_gpt_3_5_1shot)

Summary:
 和室湖畔側のお部屋は景色が良く清潔感があふれ、夕食朝食ともに大満足だという評価が多く見られます。子供たちも屋上スパで大満足だったという声が多く、家族旅行にぴったりの宿泊施設と言えそうです。食事は夕食・朝食共にバイキング形式で種類が豊富で美味しいと評価され、特に露天風呂が素晴らしいという声も多いです。全体的に清潔感があり、サービスに関しても好評が多く、再訪を希望する声も多く見られます。

GPT-3.5 Evaluation:
 Reasoning: 複数レビューの共通点（湖畔側和室の景観と清潔さ、朝夕バイキングの満足度、屋上スパの人気、サービス評価、再訪意向）を「部屋→子ども/スパ→食事→風呂→全体評価」という流れで整理しており、文同士のつながりも自然です。重複気味な箇所（スパ満足・食事満足の繰り返し）はあるものの、全体として構成が明確で読みやすいです。  
Score: 5


In [20]:
one_shot_result = summarize(reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content

In [21]:
print(one_shot_result)

総合的に素晴らしい宿泊施設との評価です。部屋は清潔で景色も良く、家族連れや友人同士で利用するには最適です。夕食、朝食ともに品数が豊富で、特に天空スパは大変満足されました。また、フロントやスタッフの対応も良く、快適な旅行となりました。再訪したいという声も多く見受けられます。


In [22]:
eval_one_shot_result = g_eval_summary(document=reviews, summary=prompt_1shot_result)
print("GPT-3.5 Evaluation:\n", eval_one_shot_result)

GPT-3.5 Evaluation:
 Reasoning: 施設（部屋・食事・屋上スパ/露天風呂・サービス）ごとに評価点と一部の不満点をまとめており、全体として話題の流れは追えます。一方で、同じ内容（満足・再訪意向）が繰り返され気味で、骨折対応など個別事例の挿入もやや唐突に感じられ、段落構成や因果のつながりが強い文章にはなっていません。それでも「良い点→一部残念点→対応面→総評」という大枠は保たれています。  
Score: 4


In [25]:
good_reviews, bad_reviews = preprocess_reviews(path='./res/reviews.json')

one_shot_bad_reviews_result = summarize(bad_reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content
eval_one_shot_result = g_eval_summary(document=bad_reviews, summary=one_shot_bad_reviews_result)

print("GPT-3.5 Evaluation:\n", eval_one_shot_result)

GPT-3.5 Evaluation:
 Reasoning: まとめ全体は「良い点が多いが不満もある」という軸で統一されており、食事・温泉・部屋・景色/家族旅行・バイキング/花火といった話題も大きくは破綻なく並んでいます。一方で、同じ内容（食事が美味しい、温泉が良い、満足）が終盤で繰り返され、結論に向けた整理（例：良い点→悪い点→総評のような構成）がやや弱く、列挙的に見える部分があります。それでも文同士のつながりは概ね自然で、読み手が全体像を追えるため中上程度の一貫性はあります。  

Score: 4


In [26]:
good_reviews, bad_reviews = preprocess_reviews(path='./res/reviews.json')

one_shot_bad_reviews_result = summarize(bad_reviews, prompt_1shot, temperature=1.0, model='gpt-3.5-turbo-0125').choices[0].message.content
eval_one_shot_result = g_eval_summary(document=bad_reviews, summary=one_shot_bad_reviews_result)

print("GPT-3.5 Evaluation:\n", eval_one_shot_result)

GPT-3.5 Evaluation:
 Reasoning: 要点（食事・温泉・景色の高評価、混雑や設備面の不満、総合的には満足）が大きくまとまっており、文同士のつながりも自然です。良い点→注意点→総評という流れで構成されていて読みやすい一方、「朝の散策」など細部がやや唐突で、全レビューの幅広い不満（例：会計ミス、コンセント不足、受付対応のばらつき等）を一文に整理しきれていないため、完璧に緻密な構成まではいきません。とはいえ全体として十分に整理されています。  
Score: 4
